# Evaluation: IAA, Precision, Recall und F1 aus der Plausibilitätsbeurteilung — **V10**

Auswertung der paarweisen **Plausibilitätsurteile** (Bögen `annotationsbogen_A/B_V10.xlsx`,
Design in `stichprobe_schluessel_V10.xlsx`). Beurteilt wird je Propositionspaar die
Modell-Entscheidung — annotierte Relation **oder** Nicht-Relation — als plausibel /
nicht plausibel. Da auch alle Nicht-Relationen der gezogenen Sätze paarweise beurteilt
werden, sind **Precision und Recall** der Extraktion bestimmbar:

| Modell-Annotation | Urteil (Gold) | Zelle |
|---|---|---|
| Relation | plausibel | **TP** |
| Relation | nicht plausibel | **FP** (falsche Relation, falscher Typ oder falsche Richtung) |
| Keine Relation | plausibel | **TN** |
| Keine Relation | nicht plausibel | **FN** (übersehene Relation) |

> Hinweis zur Einordnung: Es handelt sich um ein Plausibilitätsdesign mit sichtbarer
> Modell-Annotation (Ankereffekt möglich); das Urteil ist streng — „nicht plausibel"
> bei einer Relation umfasst auch falschen Typ oder falsche Richtung. Precision misst
> damit den Anteil vollständig haltbarer Annotationen.

**Neu in V10 (Teil C, Abschnitt 5):** Erhebung **verbundener Prämissen (R6)** — Fälle, in denen
mehrere Propositionen eine Konklusion erst *gemeinsam* stützen ({P1, P2} → P3).
Die R6-Erhebung ist **additiv**: Sie verändert die Paarmetriken oben nicht. Das Paarurteil bleibt
auch bei Verbundverdacht „plausibel", weil unter V9 zwischen den *einzelnen* Propositionen
tatsächlich keine Relation besteht. Abschnitt 5.1 prüft, ob diese Regel eingehalten wurde.

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics import cohen_kappa_score

In [2]:
# ===== KONFIGURATION =====
FILE_A   = "annotationsbogen_A_V10.xlsx"       # ausgefüllter Bogen Annotator:in A
FILE_B   = "annotationsbogen_B_V10.xlsx"       # ausgefüllter Bogen Annotator:in B
KEY_FILE = "stichprobe_schluessel_V10.xlsx"    # Design und Gewichte
OUTPUT_FILE = "intra_relationen_eval_ergebnisse.xlsx"

# Optional: adjudizierte Disagreements (Format wie Disagreement-Blatt,
# zusätzlich Spalte 'Adjudikation' mit plausibel/nicht plausibel). None = ausschließen.
ADJUDICATION_FILE = None

# Spaltennamen der R6-Blätter (falls die Bögen umbenannt werden)
R6_SHEET   = "Verbünde"
SATZ_SHEET = "Sätze"
COL_PRAEM  = "Ko-Prämissen (IDs, mit \';\' trennen)"
COL_KONKL  = "Konklusion (ID oder \'implizit\')"

## 1. Daten laden und ausrichten

In [3]:
def pnum(pid):
    return int(re.match(r"P\(\d+\.(\d+)\)", pid).group(1))

def load_bogen(path):
    df = pd.read_excel(path, sheet_name="Annotation")
    df["paar_key"] = ["|".join(sorted([a, b], key=pnum)) for a, b in zip(df["P1-ID"], df["P2-ID"])]
    df["eval"] = df["Evaluation"].astype(str).str.strip().str.lower().replace({"nan": ""})
    if "R6-Verdacht" not in df.columns:      # Abwärtskompatibilität zu V9-Bögen
        df["R6-Verdacht"] = np.nan
        df["R6-Verbund-Nr"] = np.nan
    df["r6_flag"] = df["R6-Verdacht"].astype(str).str.strip().str.lower().eq("ja")
    return df.set_index("paar_key")

A = load_bogen(FILE_A)
B = load_bogen(FILE_B)
assert set(A.index) == set(B.index), "Bögen enthalten unterschiedliche Paare!"
B = B.loc[A.index]

leer = [( "A", (A["eval"] == "").sum()), ("B", (B["eval"] == "").sum())]
for wer, n in leer:
    if n: print(f"WARNUNG: {n} unausgefüllte Urteile bei Annotator:in {wer} — Paare werden ausgeschlossen.")
mask = A["eval"].isin(["plausibel", "nicht plausibel"]) & B["eval"].isin(["plausibel", "nicht plausibel"])
A_all, B_all = A.copy(), B.copy()          # ungefiltert, für die R6-Auswertung
A, B = A[mask], B[mask]

saetze = pd.read_excel(KEY_FILE, sheet_name="Sätze").set_index("Satz-ID")
gewicht = A["Satz-ID"].map(saetze["Gewicht"])
A["modell_pos"] = A["Modell-Annotation"].astype(str).str.strip() != "Keine Relation"
print(f"{len(A)} beidseitig beurteilte Paare aus {A['Satz-ID'].nunique()} Sätzen "
      f"({int(A['modell_pos'].sum())} Relationen, {int((~A['modell_pos']).sum())} Nicht-Relationen)")

353 beidseitig beurteilte Paare aus 105 Sätzen (101 Relationen, 252 Nicht-Relationen)


## 2. Teil A: Inter-Annotator-Agreement

Cohen's κ über das Plausibilitätsurteil (gesamt sowie getrennt nach Modell-Annotation:
Relation vs. Nicht-Relation — die beiden Teilaufgaben können unterschiedlich schwer sein).

In [4]:
y1 = (A["eval"] == "plausibel").astype(int)
y2 = (B["eval"] == "plausibel").astype(int)

def iaa(mask, label):
    a, b = y1[mask], y2[mask]
    kap = cohen_kappa_score(a, b) if a.nunique() > 1 or b.nunique() > 1 else float("nan")
    print(f"{label:<28} Kappa = {kap:6.3f} | Übereinstimmung = {(a == b).mean()*100:5.1f}% | N = {mask.sum()}")
    return kap, (a == b).mean()

kappa_ges, agree_ges = iaa(pd.Series(True, index=A.index), "Gesamt")
kappa_rel, agree_rel = iaa(A["modell_pos"], "  nur Relationen")
kappa_una, agree_una = iaa(~A["modell_pos"], "  nur Nicht-Relationen")

df_dis = pd.DataFrame({
    "Satz-ID": A["Satz-ID"], "Kapitel": A["Kapitel"],
    "P1-ID": A["P1-ID"], "P1-Text": A["P1-Text"], "P2-ID": A["P2-ID"], "P2-Text": A["P2-Text"],
    "Modell-Annotation": A["Modell-Annotation"],
    "A_Urteil": A["eval"], "B_Urteil": B["eval"],
    "A_Anmerkung": A["Anmerkung"], "B_Anmerkung": B["Anmerkung"],
    "A_R6": np.where(A["r6_flag"], A["R6-Verbund-Nr"].astype(str), ""),
    "B_R6": np.where(B["r6_flag"], B["R6-Verbund-Nr"].astype(str), ""),
})[y1 != y2]
print(f"\nDisagreements: {len(df_dis)}"
      f" | davon mit R6-Verdacht bei mindestens einer Person: "
      f"{int(((df_dis['A_R6'] != '') | (df_dis['B_R6'] != '')).sum())}")

Gesamt                       Kappa =  0.524 | Übereinstimmung =  89.5% | N = 353
  nur Relationen             Kappa =  0.605 | Übereinstimmung =  84.2% | N = 101
  nur Nicht-Relationen       Kappa =  0.319 | Übereinstimmung =  91.7% | N = 252

Disagreements: 37 | davon mit R6-Verdacht bei mindestens einer Person: 1


## 3. Teil B: Goldstandard und Modellgüte

Gold = übereinstimmendes Urteil; Disagreements werden ausgeschlossen oder per
`ADJUDICATION_FILE` aufgelöst. Aus Gold × Modell-Annotation ergeben sich TP/FP/TN/FN und
damit Precision, Recall und F1 — ungewichtet und **gewichtet** (Hochrechnung über die
Schichtgewichte N/n auf den Rahmen: Sätze mit 2–6 Propositionen).

In [5]:
gold = pd.Series(np.where(y1 == y2, y1, -1), index=A.index)  # 1 = plausibel, 0 = nicht plausibel, -1 = offen

if ADJUDICATION_FILE:
    adj = pd.read_excel(ADJUDICATION_FILE)
    adj["paar_key"] = ["|".join(sorted([a, b], key=pnum)) for a, b in zip(adj["P1-ID"], adj["P2-ID"])]
    adj = adj.set_index("paar_key")["Adjudikation"].astype(str).str.strip().str.lower()
    for k, v in adj.items():
        if k in gold.index and v in ("plausibel", "nicht plausibel"):
            gold.loc[k] = 1 if v == "plausibel" else 0
    print(f"Adjudikation eingespielt: {adj.isin(['plausibel','nicht plausibel']).sum()} Urteile")

decided = gold != -1
g, w, pos = gold[decided], gewicht[decided], A.loc[decided, "modell_pos"]

TP = int((pos & (g == 1)).sum()); FP = int((pos & (g == 0)).sum())
TN = int((~pos & (g == 1)).sum()); FN = int((~pos & (g == 0)).sum())
prec = TP / (TP + FP) if TP + FP else float("nan")
rec  = TP / (TP + FN) if TP + FN else float("nan")
f1   = 2 * prec * rec / (prec + rec) if prec + rec else float("nan")
acc  = (TP + TN) / len(g)

def ws_(m): return float(w[m].sum())
wTP, wFP, wFN = ws_(pos & (g == 1)), ws_(pos & (g == 0)), ws_(~pos & (g == 0))
prec_w = wTP / (wTP + wFP) if wTP + wFP else float("nan")
rec_w  = wTP / (wTP + wFN) if wTP + wFN else float("nan")
f1_w   = 2 * prec_w * rec_w / (prec_w + rec_w) if prec_w + rec_w else float("nan")

print("=" * 64)
print("MODELLGÜTE (aus Plausibilitätsurteilen, Paar-Ebene)")
print("=" * 64)
print(f"Entschiedene Paare: {len(g)} | ausgeschlossen (Disagreement): {int((~decided).sum())}")
print(f"Konfusion: TP={TP}  FP={FP}  FN={FN}  TN={TN}")
print(f"Gesamtplausibilität (Accuracy): {acc:.3f}")
print(f"UNGEWICHTET: Precision {prec:.3f} | Recall {rec:.3f} | F1 {f1:.3f}")
print(f"GEWICHTET:   Precision {prec_w:.3f} | Recall {rec_w:.3f} | F1 {f1_w:.3f}")

MODELLGÜTE (aus Plausibilitätsurteilen, Paar-Ebene)
Entschiedene Paare: 316 | ausgeschlossen (Disagreement): 37
Konfusion: TP=65  FP=20  FN=6  TN=225
Gesamtplausibilität (Accuracy): 0.918
UNGEWICHTET: Precision 0.765 | Recall 0.915 | F1 0.833
GEWICHTET:   Precision 0.818 | Recall 0.967 | F1 0.886


In [6]:
# Aufschlüsselung: Plausibilität je Modell-Annotationstyp (Präzisionsperspektive)
A_dec = A.loc[decided].copy(); A_dec["gold"] = g
A_dec["typ_label"] = A_dec["Modell-Annotation"].astype(str).str.split(":").str[0].str.strip()
typ_rows = []
for t, grp in A_dec.groupby("typ_label"):
    n = len(grp); ok = int((grp["gold"] == 1).sum())
    typ_rows.append({"Modell-Annotation": t, "N (entschieden)": n, "plausibel": ok,
                     "Anteil plausibel": round(ok / n, 3),
                     "Hinweis": "n < 10: nicht belastbar" if n < 10 else ""})
df_typ = pd.DataFrame(typ_rows).sort_values("N (entschieden)", ascending=False)
print(df_typ.to_string(index=False))

   Modell-Annotation  N (entschieden)  plausibel  Anteil plausibel Hinweis
      Keine Relation              231        225             0.974        
              PR-BER               55         51             0.927        
              PR-FLG               16         14             0.875        
PR-INK (symmetrisch)               14          0             0.000        


## 4. Teil C: Verbundene Prämissen (R6)

Ziel dieser Erhebung ist eine **Grundrate**: In wie vielen der geprüften Sätze kommen Verbünde
tatsächlich vor? Davon hängt ab, ob Regel R6 in Spur 2 (Finetuning) aufgenommen wird oder nur als
Schemagrenze dokumentiert wird.

Verbundfähig sind nur Sätze mit mindestens 3 Propositionen. Der Nenner der Grundrate sind die
Sätze, die im Blatt `Sätze` als **geprüft** markiert wurden — nicht alle Sätze der Stichprobe.

**Grenzfälle** (Status `ja - Grenzfall (kein Verbund nach Regel)`) zählen als geprüft, aber nicht als Verbund: gemeinsame Stützung gesehen, nach den Prüffragen aber kein regelkonformer Verbund — etwa Ko-Prämisse aus dem Weltwissen, implizite Konklusion oder Ko-Prämisse erst im Nachbarsatz. Sie werden separat ausgewiesen, weil sie die Reichweite des Schemas zeigen.

In [7]:
def load_r6(path, wer):
    """Verbund-Blatt eines Bogens laden, Beispielzeile und Leerzeilen entfernen."""
    vb = pd.read_excel(path, sheet_name=R6_SHEET)
    vb = vb.dropna(subset=["Verbund-Nr", COL_PRAEM, COL_KONKL], how="any").copy()
    vb = vb[~vb["Begründung (1 Satz)"].astype(str).str.contains("BEISPIELZEILE", na=False)]
    if not len(vb):
        vb["praemissen"] = []; vb["verbund_key"] = []; vb["k"] = []; vb["Annotator"] = []
        return vb
    def parse(s):
        return sorted(re.findall(r"P\(\d+\.\d+\)", str(s)), key=pnum)
    vb["praemissen"] = vb[COL_PRAEM].map(parse)
    vb["k"] = vb["praemissen"].map(len)
    vb["konklusion"] = vb[COL_KONKL].astype(str).str.strip()
    vb["verbund_key"] = ["+".join(p) + ">" + k for p, k in zip(vb["praemissen"], vb["konklusion"])]
    vb["Annotator"] = wer
    return vb

def load_saetze(path, wer):
    sz = pd.read_excel(path, sheet_name=SATZ_SHEET)
    sz["status"] = sz["Verbund geprüft"].astype(str).str.strip()
    sz["geprueft"] = sz["status"].str.lower().str.startswith("ja")
    sz["grenzfall"] = sz["status"].str.lower().str.startswith("ja - grenzfall")
    sz["moeglich"] = sz["Anzahl"] >= 3
    sz["Annotator"] = wer
    return sz

R6_A, R6_B = load_r6(FILE_A, "A"), load_r6(FILE_B, "B")
SZ_A, SZ_B = load_saetze(FILE_A, "A"), load_saetze(FILE_B, "B")

print(f"Verbundfähige Sätze (>= 3 Propositionen): {int(SZ_A['moeglich'].sum())} von {len(SZ_A)}")
for wer, sz, r6 in (("A", SZ_A, R6_A), ("B", SZ_B, R6_B)):
    gepr = int((sz["moeglich"] & sz["geprueft"]).sum()); mgl = int(sz["moeglich"].sum())
    print(f"  {wer}: {gepr}/{mgl} Sätze geprüft ({gepr/mgl*100:4.1f} %) | "
          f"{len(r6)} Verbünde in {r6['Satz-ID'].nunique() if len(r6) else 0} Sätzen")
    vc = sz.loc[sz["moeglich"], "status"].replace({"nan": "(leer)", "": "(leer)"}).value_counts()
    print("     Status: " + " · ".join(f"{i}: {n}" for i, n in vc.items()))
    if gepr < mgl:
        print(f"     HINWEIS: {mgl-gepr} verbundfähige Sätze ohne Eintrag in 'Verbund geprüft' — "
              f"die Grundrate unten ist dann eine Untergrenze.")

Verbundfähige Sätze (>= 3 Propositionen): 56 von 105
  A: 0/56 Sätze geprüft ( 0.0 %) | 0 Verbünde in 0 Sätzen
     Status: (leer): 56
     HINWEIS: 56 verbundfähige Sätze ohne Eintrag in 'Verbund geprüft' — die Grundrate unten ist dann eine Untergrenze.
  B: 56/56 Sätze geprüft (100.0 %) | 5 Verbünde in 5 Sätzen
     Status: ja - kein Verbund gefunden: 51 · ja - Verbund eingetragen: 3 · ja - Grenzfall (kein Verbund nach Regel): 2


### 4.1 Konsistenzprüfungen

Diese Prüfungen laufen still durch, wenn alles stimmt.

In [8]:
probleme = []

for wer, df_all, r6, sz in (("A", A_all, R6_A, SZ_A), ("B", B_all, R6_B, SZ_B)):
    # (1) Regelverstoß: Verbundverdacht UND 'nicht plausibel' -> verfälscht den Recall
    v = df_all[df_all["r6_flag"] & (df_all["eval"] == "nicht plausibel")]
    for k, r in v.iterrows():
        probleme.append({"Art": "Urteil trotz R6-Verdacht 'nicht plausibel'", "Annotator": wer,
                         "Satz-ID": r["Satz-ID"], "Detail": f"{r['P1-ID']}|{r['P2-ID']} -> {r['R6-Verbund-Nr']}"})
    # (2) Markierung ohne Verbund-Nr / ohne Eintrag im Verbund-Blatt
    nummern = set(r6["Verbund-Nr"].astype(str)) if len(r6) else set()
    for k, r in df_all[df_all["r6_flag"]].iterrows():
        nr = str(r["R6-Verbund-Nr"]).strip()
        if nr in ("", "nan"):
            probleme.append({"Art": "R6-Verdacht ohne Verbund-Nr", "Annotator": wer,
                             "Satz-ID": r["Satz-ID"], "Detail": f"{r['P1-ID']}|{r['P2-ID']}"})
        elif nr not in nummern:
            probleme.append({"Art": "Verbund-Nr ohne Zeile im Blatt 'Verbünde'", "Annotator": wer,
                             "Satz-ID": r["Satz-ID"], "Detail": nr})
    # (3) Verbund-Zeilen gegen die Propositionen des Satzes prüfen
    props_je_satz = {r["Satz-ID"]: set(re.findall(r"P\(\d+\.\d+\)", str(r["Propositionen des Satzes"])))
                     for _, r in sz.iterrows()}
    for _, r in r6.iterrows():
        sid, ids = r["Satz-ID"], set(r["praemissen"])
        bekannt = props_je_satz.get(sid, set())
        if r["k"] < 2:
            probleme.append({"Art": "Verbund mit weniger als 2 Ko-Prämissen", "Annotator": wer,
                             "Satz-ID": sid, "Detail": r["Verbund-Nr"]})
        if r["konklusion"] in ids:
            probleme.append({"Art": "Konklusion steht auch unter den Ko-Prämissen", "Annotator": wer,
                             "Satz-ID": sid, "Detail": r["Verbund-Nr"]})
        unbekannt = (ids | ({r["konklusion"]} if r["konklusion"].startswith("P(") else set())) - bekannt
        if unbekannt and r["Lage"] == "im Satz":
            probleme.append({"Art": "ID gehört nicht zu diesem Satz", "Annotator": wer,
                             "Satz-ID": sid, "Detail": f"{r['Verbund-Nr']}: {sorted(unbekannt)}"})
        if not sz.loc[sz["Satz-ID"] == sid, "geprueft"].any():
            probleme.append({"Art": "Verbund eingetragen, Satz aber nicht als geprüft markiert",
                             "Annotator": wer, "Satz-ID": sid, "Detail": r["Verbund-Nr"]})

df_probleme = pd.DataFrame(probleme)
if len(df_probleme):
    print(f"{len(df_probleme)} Hinweise — bitte vor der Auswertung klären:\n")
    print(df_probleme.to_string(index=False))
else:
    print("Keine Auffälligkeiten.")

2 Hinweise — bitte vor der Auswertung klären:

                                   Art Annotator Satz-ID    Detail
Verbund mit weniger als 2 Ko-Prämissen         B  S-8230 S-8230-V1
Verbund mit weniger als 2 Ko-Prämissen         B  S-2493 S-2493-V1


### 4.2 Grundrate und Häufigkeitsverteilung

Die gewichtete Grundrate rechnet über die Schichtgewichte auf den Rahmen hoch (Sätze mit 2–6
Propositionen) und beantwortet damit die eigentliche Frage: Wie häufig sind Verbünde im Korpus?

In [9]:
def grundrate(r6, sz, wer):
    basis = sz[sz["moeglich"] & sz["geprueft"]]
    mit = set(r6["Satz-ID"]) & set(basis["Satz-ID"])
    n, k = len(basis), len(mit)
    gew = basis["Satz-ID"].map(saetze["Gewicht"])
    gew_k = basis.loc[basis["Satz-ID"].isin(mit), "Satz-ID"].map(saetze["Gewicht"]).sum()
    return {"Annotator": wer, "geprüfte Sätze": n, "Sätze mit Verbund": k,
            "Grundrate": round(k / n, 4) if n else float("nan"),
            "Grundrate gewichtet": round(gew_k / gew.sum(), 4) if n and gew.sum() else float("nan"),
            "Grenzfälle (Schemagrenze)": int((basis["Satz-ID"].isin(sz.loc[sz["grenzfall"], "Satz-ID"])).sum()),
            "Verbünde gesamt": len(r6),
            "davon 'sicher'": int((r6["Sicherheit"].astype(str).str.strip().str.lower() == "sicher").sum()) if len(r6) else 0}

df_rate = pd.DataFrame([grundrate(R6_A, SZ_A, "A"), grundrate(R6_B, SZ_B, "B")])
print(df_rate.to_string(index=False))

R6 = pd.concat([R6_A, R6_B], ignore_index=True)
if len(R6):
    print("\nVerteilungen (beide Bögen zusammen, Verbünde einzeln gezählt):")
    for sp in ["k", "Typ", "Art der Ko-Prämissen", "Lage", "Sicherheit"]:
        vc = R6[sp].value_counts(dropna=False)
        print(f"  {sp:<22}" + " · ".join(f"{i}: {n}" for i, n in vc.items()))
    print("\nHinweis zu 'Art der Ko-Prämissen': Zählt ein Fall als 'Brückenprinzip', ist er nach "
          "Brandom keine echte Verbundinferenz, sondern eine explizit gemachte Paarinferenz.")
else:
    print("\nKeine Verbünde eingetragen — R6 wäre in diesem Material leer.")

Annotator  geprüfte Sätze  Sätze mit Verbund  Grundrate  Grundrate gewichtet  Grenzfälle (Schemagrenze)  Verbünde gesamt  davon 'sicher'
        A               0                  0        NaN                  NaN                          0                0               0
        B              56                  5     0.0893               0.0808                          2                5               2

Verteilungen (beide Bögen zusammen, Verbünde einzeln gezählt):
  k                     2.0: 3 · 1.0: 2
  Typ                   BER: 5
  Art der Ko-Prämissen  substantiell: 2 · gemischt: 2 · Brückenprinzip: 1
  Lage                  im Satz: 3 · nan: 2
  Sicherheit            unsicher: 3 · sicher: 2

Hinweis zu 'Art der Ko-Prämissen': Zählt ein Fall als 'Brückenprinzip', ist er nach Brandom keine echte Verbundinferenz, sondern eine explizit gemachte Paarinferenz.


### 4.3 Übereinstimmung auf Verbundebene

Verglichen wird über `verbund_key` (sortierte Ko-Prämissen + Konklusion). Basis sind nur die
Sätze, die **beide** Personen geprüft haben; sonst wäre eine fehlende Prüfung nicht von einem
fehlenden Verbund zu unterscheiden.

In [10]:
basis = set(SZ_A.loc[SZ_A["moeglich"] & SZ_A["geprueft"], "Satz-ID"]) & \
        set(SZ_B.loc[SZ_B["moeglich"] & SZ_B["geprueft"], "Satz-ID"])
kA = set(R6_A.loc[R6_A["Satz-ID"].isin(basis), "verbund_key"]) if len(R6_A) else set()
kB = set(R6_B.loc[R6_B["Satz-ID"].isin(basis), "verbund_key"]) if len(R6_B) else set()

beide, nur_a, nur_b = kA & kB, kA - kB, kB - kA
jacc = len(beide) / len(kA | kB) if (kA | kB) else float("nan")
print(f"Gemeinsam geprüfte Sätze: {len(basis)}")
print(f"Verbünde: beide {len(beide)} | nur A {len(nur_a)} | nur B {len(nur_b)} | Jaccard {jacc:.3f}")
if (kA | kB) and len(beide) < len(kA | kB):
    print("\nNicht übereinstimmende Verbünde (Kandidaten für die Adjudikation):")
    for k in sorted(nur_a | nur_b):
        print(f"  {'nur A' if k in nur_a else 'nur B'}: {k}")

# Teilübereinstimmung: gleiche Konklusion, abweichende Prämissenmenge
if len(R6_A) and len(R6_B):
    konkl_a = {(r["Satz-ID"], r["konklusion"]): set(r["praemissen"]) for _, r in R6_A.iterrows()}
    konkl_b = {(r["Satz-ID"], r["konklusion"]): set(r["praemissen"]) for _, r in R6_B.iterrows()}
    teil = [(k, sorted(konkl_a[k]), sorted(konkl_b[k])) for k in set(konkl_a) & set(konkl_b)
            if konkl_a[k] != konkl_b[k]]
    if teil:
        print("\nGleiche Konklusion, abweichende Prämissenmenge:")
        for k, a_, b_ in teil: print(f"  {k}: A {a_} vs. B {b_}")

Gemeinsam geprüfte Sätze: 0
Verbünde: beide 0 | nur A 0 | nur B 0 | Jaccard nan


### 4.4 Sensitivität: Was hängt an R6?

Zwei Kennzahlen zur Einordnung des Befunds:

1. Wie viele der als **FN** gewerteten Paare tragen einen R6-Verdacht? Sie wären unter einem
   Verbundregime gar keine übersehenen Paarrelationen, sondern Teile eines Verbunds.
2. Wie ändern sich Precision und Recall, wenn diese Paare ausgeschlossen werden? Das ist eine
   reine Kontrollrechnung; die berichteten Zahlen bleiben die aus Abschnitt 3.

In [11]:
r6_pair = A_all["r6_flag"] | B_all["r6_flag"]
fn_mask = (~A_dec["modell_pos"]) & (A_dec["gold"] == 0)
fn_r6 = int(r6_pair.reindex(A_dec.index).fillna(False)[fn_mask].sum())
print(f"FN gesamt: {FN} | davon mit R6-Verdacht: {fn_r6}")

keep = ~r6_pair.reindex(A_dec.index).fillna(False)
g2, pos2 = A_dec.loc[keep, "gold"], A_dec.loc[keep, "modell_pos"]
TP2 = int((pos2 & (g2 == 1)).sum()); FP2 = int((pos2 & (g2 == 0)).sum()); FN2 = int((~pos2 & (g2 == 0)).sum())
prec2 = TP2 / (TP2 + FP2) if TP2 + FP2 else float("nan")
rec2  = TP2 / (TP2 + FN2) if TP2 + FN2 else float("nan")
print(f"Kontrollrechnung ohne R6-markierte Paare (N={int(keep.sum())}): "
      f"Precision {prec2:.3f} (Δ {prec2-prec:+.3f}) | Recall {rec2:.3f} (Δ {rec2-rec:+.3f})")
print("\nEntscheidungshilfe: Ist die Grundrate klein und sind die Δ-Werte nahe null, "
      "verliert das Paarregime durch den Verzicht auf R6 kaum etwas.")

FN gesamt: 6 | davon mit R6-Verdacht: 0
Kontrollrechnung ohne R6-markierte Paare (N=312): Precision 0.765 (Δ +0.000) | Recall 0.915 (Δ +0.000)

Entscheidungshilfe: Ist die Grundrate klein und sind die Δ-Werte nahe null, verliert das Paarregime durch den Verzicht auf R6 kaum etwas.


## 5. Export

In [12]:
from openpyxl.styles import Font, PatternFill, Alignment

def paarliste(mask, art):
    sub = A_dec[mask]
    return pd.DataFrame({
        "Art": art, "Satz-ID": sub["Satz-ID"], "Kapitel": sub["Kapitel"],
        "P1-ID": sub["P1-ID"], "P1-Text": sub["P1-Text"], "P2-ID": sub["P2-ID"], "P2-Text": sub["P2-Text"],
        "Modell-Annotation": sub["Modell-Annotation"],
        "A_Anmerkung": sub["Anmerkung"], "B_Anmerkung": B.loc[sub.index, "Anmerkung"],
        "R6-Verdacht": np.where(r6_pair.reindex(sub.index).fillna(False), "ja", ""),
    })
df_fehler = pd.concat([
    paarliste(A_dec["modell_pos"] & (A_dec["gold"] == 0), "FP (Annotation nicht plausibel)"),
    paarliste(~A_dec["modell_pos"] & (A_dec["gold"] == 0), "FN (übersehene Relation)"),
], ignore_index=True)

df_r6 = pd.concat([R6_A, R6_B], ignore_index=True)
if len(df_r6):
    df_r6 = df_r6.drop(columns=["praemissen"]).assign(
        Übereinstimmung=lambda d: np.where(d["verbund_key"].isin(beide), "beide",
                                  np.where(d["Satz-ID"].isin(basis), "nur eine Person", "Satz nicht beidseitig geprüft")))

summary = pd.DataFrame([
    {"Metrik": "Inter-Annotator Agreement", "Wert": "", "Detail": ""},
    {"Metrik": "  Cohen's Kappa (gesamt)", "Wert": f"{kappa_ges:.3f}", "Detail": f"{agree_ges*100:.1f}% Übereinstimmung, N={len(y1)}"},
    {"Metrik": "  Kappa nur Relationen", "Wert": f"{kappa_rel:.3f}", "Detail": f"{agree_rel*100:.1f}%, N={int(A['modell_pos'].sum())}"},
    {"Metrik": "  Kappa nur Nicht-Relationen", "Wert": f"{kappa_una:.3f}", "Detail": f"{agree_una*100:.1f}%, N={int((~A['modell_pos']).sum())}"},
    {"Metrik": "  Disagreements", "Wert": str(len(df_dis)), "Detail": "ausgeschlossen bzw. adjudiziert (s. Blatt)"},
    {"Metrik": "", "Wert": "", "Detail": ""},
    {"Metrik": "Modellgüte (Paar-Ebene)", "Wert": "", "Detail": ""},
    {"Metrik": "  Gesamtplausibilität (Accuracy)", "Wert": f"{acc:.3f}", "Detail": f"entschiedene Paare: {len(g)}"},
    {"Metrik": "  Precision", "Wert": f"{prec:.3f}", "Detail": f"TP={TP}, FP={FP} (streng: inkl. Typ-/Richtungsfehler)"},
    {"Metrik": "  Recall", "Wert": f"{rec:.3f}", "Detail": f"FN={FN} (übersehene Relationen), TN={TN}"},
    {"Metrik": "  F1", "Wert": f"{f1:.3f}", "Detail": ""},
    {"Metrik": "  Precision / Recall / F1 (gewichtet)", "Wert": f"{prec_w:.3f} / {rec_w:.3f} / {f1_w:.3f}",
     "Detail": "Hochrechnung auf Rahmen (Sätze mit 2–6 Propositionen)"},
    {"Metrik": "", "Wert": "", "Detail": ""},
    {"Metrik": "Verbundene Prämissen (R6, additiv)", "Wert": "", "Detail": ""},
    {"Metrik": "  Verbundfähige Sätze", "Wert": str(int(SZ_A["moeglich"].sum())),
     "Detail": f"von {len(SZ_A)} Sätzen der Stichprobe (>= 3 Propositionen)"},
    {"Metrik": "  Geprüft A / B", "Wert": f"{int((SZ_A['moeglich'] & SZ_A['geprueft']).sum())} / {int((SZ_B['moeglich'] & SZ_B['geprueft']).sum())}",
     "Detail": "Nenner der Grundrate"},
    {"Metrik": "  Verbünde A / B", "Wert": f"{len(R6_A)} / {len(R6_B)}", "Detail": "Zeilen im Blatt 'Verbünde'"},
    {"Metrik": "  Grundrate A / B", "Wert": " / ".join(f"{v:.3f}" if v == v else "n/a" for v in df_rate["Grundrate"]),
     "Detail": "Anteil geprüfter Sätze mit mindestens einem Verbund"},
    {"Metrik": "  Grundrate gewichtet A / B", "Wert": " / ".join(f"{v:.3f}" if v == v else "n/a" for v in df_rate["Grundrate gewichtet"]),
     "Detail": "Hochrechnung auf den Rahmen"},
    {"Metrik": "  Grenzfälle A / B", "Wert": " / ".join(str(v) for v in df_rate["Grenzfälle (Schemagrenze)"]),
     "Detail": "geprüft, aber kein regelkonformer Verbund (Weltwissen, implizite Konklusion, Nachbarsatz)"},
    {"Metrik": "  Übereinstimmung (Jaccard)", "Wert": f"{jacc:.3f}" if jacc == jacc else "n/a",
     "Detail": f"beide {len(beide)} | nur A {len(nur_a)} | nur B {len(nur_b)}"},
    {"Metrik": "  FN mit R6-Verdacht", "Wert": f"{fn_r6} von {FN}", "Detail": "Kontrollrechnung, s. Blatt 'R6-Verbünde'"},
    {"Metrik": "  Konsistenzhinweise", "Wert": str(len(df_probleme)), "Detail": "s. Blatt 'R6-Prüfhinweise'"},
    {"Metrik": "", "Wert": "", "Detail": ""},
    {"Metrik": "Stichprobe", "Wert": "", "Detail": ""},
    {"Metrik": "  Design", "Wert": f"{A['Satz-ID'].nunique()} Sätze / {len(A)} Paare",
     "Detail": "geschichtet Kapitel × Modell-Signal (Seed s. Schlüsseldatei); Urteil: plausibel/nicht plausibel je Paar; Modell-Annotation sichtbar (Anker möglich)"},
])

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary.to_excel(writer, index=False, sheet_name="Zusammenfassung")
    df_typ.to_excel(writer, index=False, sheet_name="Plausibilität pro Typ")
    if len(df_fehler): df_fehler.to_excel(writer, index=False, sheet_name="Fehler (FP-FN)")
    if len(df_dis): df_dis.to_excel(writer, index=False, sheet_name="Disagreements")
    df_rate.to_excel(writer, index=False, sheet_name="R6-Grundrate")
    if len(df_r6): df_r6.to_excel(writer, index=False, sheet_name="R6-Verbünde")
    if len(df_probleme): df_probleme.to_excel(writer, index=False, sheet_name="R6-Prüfhinweise")
    hf = PatternFill(start_color="2B4C7E", end_color="2B4C7E", fill_type="solid")
    for ws in writer.sheets.values():
        for cell in ws[1]:
            cell.fill = hf; cell.font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        ws.auto_filter.ref = ws.dimensions
print(f"Exportiert: {OUTPUT_FILE}")

Exportiert: intra_relationen_eval_ergebnisse.xlsx
